# RAG Fundamentals

**Module 10: Retrieval-Augmented Generation**  
**Objective**: Build RAG systems from scratch

RAG combines retrieval with generation:
- Vector databases for semantic search
- Embedding models
- Similarity metrics
- Retrieval strategies
- Context integration

## What You'll Learn
1. Vector embeddings and similarity
2. Building vector database from scratch
3. Semantic search algorithms
4. Chunking strategies for documents
5. Hybrid search (dense + sparse)
6. Evaluation of retrieval quality

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from typing import List, Dict, Tuple
import re
from collections import Counter

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}\n")

torch.manual_seed(42)
np.random.seed(42)

## 1. What is RAG?

**Problem**: LLMs have limitations
- Knowledge cutoff (training data ends at specific date)
- Hallucinations (make up facts)
- No access to private/proprietary data

**Solution**: Retrieval-Augmented Generation
1. Retrieve relevant documents from knowledge base
2. Provide as context to LLM
3. Generate answer grounded in retrieved facts

**Example**:
```
Query: "What did the CEO say about Q4 revenue?"

Retrieved: "In our Q4 2023 earnings call, CEO mentioned revenue grew 25% YoY to $500M..."

LLM Answer: "According to the Q4 2023 earnings call, the CEO stated revenue grew 25% year-over-year, reaching $500 million..."
```

In [ ]:
def visualize_rag_pipeline():
    """Visualize RAG pipeline."""
    
    fig, ax = plt.subplots(figsize=(14, 8))
    ax.axis('off')
    
    # Components
    components = [
        ("User Query", (0.1, 0.9), 'lightblue'),
        ("Embedding\nModel", (0.3, 0.9), 'lightgreen'),
        ("Vector\nDatabase", (0.3, 0.6), 'lightyellow'),
        ("Retrieved\nDocuments", (0.5, 0.6), 'lightcoral'),
        ("LLM +\nContext", (0.7, 0.75), 'plum'),
        ("Final\nAnswer", (0.9, 0.75), 'lightblue')
    ]
    
    for label, (x, y), color in components:
        circle = plt.Circle((x, y), 0.08, color=color, ec='black', linewidth=2)
        ax.add_patch(circle)
        ax.text(x, y, label, ha='center', va='center', fontsize=10, weight='bold')
    
    # Arrows
    arrows = [
        ((0.18, 0.9), (0.22, 0.9)),  # Query -> Embedding
        ((0.3, 0.82), (0.3, 0.68)),  # Embedding -> DB
        ((0.38, 0.6), (0.42, 0.6)),  # DB -> Retrieved
        ((0.58, 0.63), (0.65, 0.70)),  # Retrieved -> LLM
        ((0.78, 0.75), (0.82, 0.75)),  # LLM -> Answer
    ]
    
    for start, end in arrows:
        ax.annotate('', xy=end, xytext=start,
                   arrowprops=dict(arrowstyle='->', lw=2, color='black'))
    
    # Add notes
    ax.text(0.3, 0.45, 'Indexed documents\nwith embeddings', 
           ha='center', fontsize=9, style='italic', bbox=dict(boxstyle='round', facecolor='wheat'))
    
    ax.text(0.7, 0.55, 'Context: Retrieved docs\nQuery: User question', 
           ha='center', fontsize=9, style='italic', bbox=dict(boxstyle='round', facecolor='wheat'))
    
    ax.set_xlim(0, 1)
    ax.set_ylim(0.3, 1)
    ax.set_title('RAG Pipeline', fontsize=16, weight='bold', pad=20)
    
    plt.tight_layout()
    plt.show()
    
    print("\nRAG Pipeline Steps:")
    print("1. User Query: 'What is the capital of France?'")
    print("2. Embed Query: Convert to dense vector [0.23, -0.45, ...]")
    print("3. Search Database: Find similar document vectors (cosine similarity)")
    print("4. Retrieve Docs: 'Paris is the capital and largest city of France...'")
    print("5. Augment Prompt: Combine query + retrieved context")
    print("6. Generate: LLM produces answer based on provided facts")
    print("\n✓ Answer is grounded in retrieved knowledge!")

visualize_rag_pipeline()

## 2. Vector Embeddings

**Core concept**: Text → Dense vectors that capture semantic meaning

Embeddings encode text so semantically similar text has similar vectors.

In [ ]:
class SimpleEmbeddingModel(nn.Module):
    """
    Simple embedding model (bag-of-words → dense vector).
    
    In practice, use models like:
    - sentence-transformers/all-MiniLM-L6-v2
    - OpenAI text-embedding-ada-002
    - Cohere embed-english-v3.0
    """
    
    def __init__(self, vocab_size=10000, embedding_dim=384):
        super().__init__()
        
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.projection = nn.Linear(embedding_dim, embedding_dim)
        
    def forward(self, token_ids):
        # token_ids: (batch, seq_len)
        embedded = self.embedding(token_ids)  # (batch, seq_len, dim)
        
        # Mean pooling
        pooled = embedded.mean(dim=1)  # (batch, dim)
        
        # Project and normalize
        output = self.projection(pooled)
        output = F.normalize(output, p=2, dim=1)  # L2 normalization
        
        return output

# Create embedding model
model = SimpleEmbeddingModel().to(device)

# Example: Embed some texts
texts_tokens = torch.randint(0, 10000, (3, 20)).to(device)
embeddings = model(texts_tokens)

print(f"Input shape: {texts_tokens.shape}")
print(f"Embedding shape: {embeddings.shape}")
print(f"Embedding norms: {embeddings.norm(dim=1)}")  # Should be ~1.0 after normalization

## 3. Similarity Metrics

In [ ]:
def compare_similarity_metrics():
    """Compare different similarity metrics."""
    
    # Create sample embeddings
    query_embedding = torch.tensor([0.8, 0.6, 0.1], dtype=torch.float32)
    
    doc_embeddings = torch.tensor([
        [0.9, 0.5, 0.1],  # Similar direction
        [-0.8, -0.6, -0.1],  # Opposite direction
        [0.1, 0.9, 0.2],  # Orthogonal
        [0.8, 0.6, 0.1],  # Identical
    ], dtype=torch.float32)
    
    # 1. Cosine Similarity
    query_norm = F.normalize(query_embedding.unsqueeze(0), p=2, dim=1)
    docs_norm = F.normalize(doc_embeddings, p=2, dim=1)
    cosine_sim = (query_norm @ docs_norm.T).squeeze()
    
    # 2. Dot Product (assuming normalized)
    dot_product = (query_norm @ docs_norm.T).squeeze()
    
    # 3. Euclidean Distance
    euclidean_dist = torch.cdist(query_embedding.unsqueeze(0), doc_embeddings, p=2).squeeze()
    
    # 4. Manhattan Distance (L1)
    manhattan_dist = torch.cdist(query_embedding.unsqueeze(0), doc_embeddings, p=1).squeeze()
    
    # Visualize
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    doc_labels = ['Similar', 'Opposite', 'Orthogonal', 'Identical']
    
    # Cosine similarity
    axes[0, 0].barh(doc_labels, cosine_sim.numpy(), color='skyblue')
    axes[0, 0].set_xlabel('Cosine Similarity')
    axes[0, 0].set_title('Cosine Similarity (higher = more similar)')
    axes[0, 0].set_xlim(-1.1, 1.1)
    axes[0, 0].axvline(0, color='red', linestyle='--', alpha=0.5)
    axes[0, 0].grid(axis='x', alpha=0.3)
    
    # Dot product
    axes[0, 1].barh(doc_labels, dot_product.numpy(), color='lightgreen')
    axes[0, 1].set_xlabel('Dot Product')
    axes[0, 1].set_title('Dot Product (higher = more similar)')
    axes[0, 1].grid(axis='x', alpha=0.3)
    
    # Euclidean distance
    axes[1, 0].barh(doc_labels, euclidean_dist.numpy(), color='lightcoral')
    axes[1, 0].set_xlabel('Euclidean Distance')
    axes[1, 0].set_title('Euclidean Distance (lower = more similar)')
    axes[1, 0].grid(axis='x', alpha=0.3)
    
    # Manhattan distance
    axes[1, 1].barh(doc_labels, manhattan_dist.numpy(), color='plum')
    axes[1, 1].set_xlabel('Manhattan Distance')
    axes[1, 1].set_title('Manhattan Distance (lower = more similar)')
    axes[1, 1].grid(axis='x', alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\nSimilarity Metrics:")
    print("="*60)
    print("\n1. COSINE SIMILARITY")
    print("   Formula: cos(θ) = (A · B) / (||A|| ||B||)")
    print("   Range: [-1, 1], 1=identical, 0=orthogonal, -1=opposite")
    print("   Use: Most common for normalized embeddings")
    
    print("\n2. DOT PRODUCT")
    print("   Formula: A · B = Σ(a_i * b_i)")
    print("   Range: (-∞, ∞), higher = more similar")
    print("   Use: Fast, equivalent to cosine if vectors normalized")
    
    print("\n3. EUCLIDEAN DISTANCE")
    print("   Formula: ||A - B|| = sqrt(Σ(a_i - b_i)²)")
    print("   Range: [0, ∞), lower = more similar")
    print("   Use: When magnitude matters")
    
    print("\n4. MANHATTAN DISTANCE")
    print("   Formula: Σ|a_i - b_i|")
    print("   Range: [0, ∞), lower = more similar")
    print("   Use: Faster than Euclidean, less weight on outliers")
    print("="*60)
    
    print("\n✓ For RAG: Cosine similarity or dot product (after normalization)")

compare_similarity_metrics()

## 4. Vector Database from Scratch

In [ ]:
class SimpleVectorDB:
    """
    Simple vector database with exact search.
    
    Real implementations (FAISS, Pinecone, Weaviate) use approximate methods
    (HNSW, IVF) for billion-scale search.
    """
    
    def __init__(self, embedding_dim=384):
        self.embedding_dim = embedding_dim
        self.vectors = []
        self.metadata = []
        self.ids = []
        
    def add(self, vectors: torch.Tensor, metadata: List[Dict], ids: List[str]):
        """
        Add vectors to database.
        
        Args:
            vectors: (N, embedding_dim) tensor
            metadata: List of metadata dicts
            ids: List of unique IDs
        """
        assert vectors.shape[1] == self.embedding_dim
        assert len(vectors) == len(metadata) == len(ids)
        
        self.vectors.append(vectors.cpu())
        self.metadata.extend(metadata)
        self.ids.extend(ids)
    
    def search(self, query_vector: torch.Tensor, top_k: int = 5) -> List[Dict]:
        """
        Search for most similar vectors.
        
        Args:
            query_vector: (embedding_dim,) query vector
            top_k: Number of results to return
            
        Returns:
            List of {id, score, metadata} dicts
        """
        if len(self.vectors) == 0:
            return []
        
        # Concatenate all vectors
        all_vectors = torch.cat(self.vectors, dim=0)  # (N, dim)
        
        # Normalize
        query_norm = F.normalize(query_vector.unsqueeze(0), p=2, dim=1)
        vectors_norm = F.normalize(all_vectors, p=2, dim=1)
        
        # Compute cosine similarity
        similarities = (query_norm @ vectors_norm.T).squeeze()  # (N,)
        
        # Get top-k
        top_scores, top_indices = torch.topk(similarities, min(top_k, len(similarities)))
        
        # Format results
        results = []
        for score, idx in zip(top_scores, top_indices):
            results.append({
                'id': self.ids[idx.item()],
                'score': score.item(),
                'metadata': self.metadata[idx.item()]
            })
        
        return results
    
    def __len__(self):
        return len(self.ids)

# Create vector database
db = SimpleVectorDB(embedding_dim=384)

# Add some documents
num_docs = 1000
doc_vectors = torch.randn(num_docs, 384)
doc_vectors = F.normalize(doc_vectors, p=2, dim=1)  # Normalize

doc_metadata = [{'text': f'Document {i}', 'source': f'source_{i%10}.pdf'} 
                for i in range(num_docs)]
doc_ids = [f'doc_{i}' for i in range(num_docs)]

db.add(doc_vectors, doc_metadata, doc_ids)

print(f"Database size: {len(db)} documents")

# Search
query = torch.randn(384)
query = F.normalize(query, p=2, dim=1)

results = db.search(query, top_k=5)

print("\nTop 5 search results:")
for i, result in enumerate(results, 1):
    print(f"{i}. ID: {result['id']}, Score: {result['score']:.4f}, Text: {result['metadata']['text']}")

## 5. Document Chunking Strategies

**Problem**: Documents are too long for embedding models (512-1024 token limit)

**Solution**: Split into chunks

In [ ]:
class DocumentChunker:
    """Split documents into chunks."""
    
    def __init__(self, chunk_size=512, overlap=50):
        """
        Args:
            chunk_size: Maximum tokens per chunk
            overlap: Tokens to overlap between chunks
        """
        self.chunk_size = chunk_size
        self.overlap = overlap
    
    def chunk_by_tokens(self, text: str) -> List[str]:
        """Chunk by token count (simple word-based approximation)."""
        words = text.split()
        chunks = []
        
        i = 0
        while i < len(words):
            chunk_words = words[i:i + self.chunk_size]
            chunks.append(' '.join(chunk_words))
            i += self.chunk_size - self.overlap
        
        return chunks
    
    def chunk_by_sentences(self, text: str) -> List[str]:
        """Chunk by sentences (semantic boundary)."""
        # Simple sentence split
        sentences = re.split(r'[.!?]+', text)
        sentences = [s.strip() for s in sentences if s.strip()]
        
        chunks = []
        current_chunk = []
        current_length = 0
        
        for sentence in sentences:
            sentence_length = len(sentence.split())
            
            if current_length + sentence_length > self.chunk_size and current_chunk:
                # Save current chunk
                chunks.append(' '.join(current_chunk))
                # Keep overlap
                overlap_sentences = current_chunk[-2:] if len(current_chunk) >= 2 else current_chunk
                current_chunk = overlap_sentences
                current_length = sum(len(s.split()) for s in current_chunk)
            
            current_chunk.append(sentence)
            current_length += sentence_length
        
        if current_chunk:
            chunks.append(' '.join(current_chunk))
        
        return chunks
    
    def chunk_by_paragraphs(self, text: str) -> List[str]:
        """Chunk by paragraphs."""
        paragraphs = text.split('\n\n')
        paragraphs = [p.strip() for p in paragraphs if p.strip()]
        
        chunks = []
        current_chunk = []
        current_length = 0
        
        for para in paragraphs:
            para_length = len(para.split())
            
            if current_length + para_length > self.chunk_size and current_chunk:
                chunks.append('\n\n'.join(current_chunk))
                current_chunk = []
                current_length = 0
            
            current_chunk.append(para)
            current_length += para_length
        
        if current_chunk:
            chunks.append('\n\n'.join(current_chunk))
        
        return chunks

# Example document
sample_doc = """
Machine learning is a subset of artificial intelligence. It focuses on teaching computers to learn from data.

Deep learning is a type of machine learning. It uses neural networks with multiple layers. These networks can learn complex patterns in data.

Natural language processing enables computers to understand human language. It combines linguistics and machine learning.

Computer vision helps computers understand images and videos. It uses convolutional neural networks for image recognition tasks.
"""

chunker = DocumentChunker(chunk_size=20, overlap=5)

# Different chunking strategies
token_chunks = chunker.chunk_by_tokens(sample_doc)
sentence_chunks = chunker.chunk_by_sentences(sample_doc)
para_chunks = chunker.chunk_by_paragraphs(sample_doc)

print("Chunking Strategies:\n")
print("="*70)

print("\n1. TOKEN-BASED CHUNKS (fixed size, may break sentences)")
for i, chunk in enumerate(token_chunks, 1):
    print(f"  Chunk {i}: {chunk[:60]}...")

print("\n2. SENTENCE-BASED CHUNKS (semantic boundaries)")
for i, chunk in enumerate(sentence_chunks, 1):
    print(f"  Chunk {i}: {chunk[:60]}...")

print("\n3. PARAGRAPH-BASED CHUNKS (natural divisions)")
for i, chunk in enumerate(para_chunks, 1):
    print(f"  Chunk {i}: {chunk[:60]}...")

print("\n" + "="*70)
print("\nBest Practice: Use sentence or paragraph chunking for better semantic coherence")

## Summary

You've mastered RAG fundamentals:
- ✅ RAG pipeline architecture
- ✅ Vector embeddings and similarity metrics
- ✅ Vector database from scratch
- ✅ Document chunking strategies
- ✅ Hybrid search concepts

**Key Insights**:
1. **Embeddings**: Normalize vectors, use cosine similarity
2. **Chunking**: Sentence/paragraph-based better than fixed tokens
3. **Search**: Exact O(N), approximate O(log N) with HNSW
4. **Hybrid**: Combine dense (semantic) + sparse (keyword) for best results

**Best Practices**:
1. Normalize all embeddings (unit vectors)
2. Use overlap in chunking (50-100 tokens)
3. Combine dense + BM25 for production
4. Monitor retrieval quality (precision@k, recall@k)

**Next Notebook**: Building production RAG systems with LangChain

## Exercises

1. **Test chunking**: Compare chunk sizes (256, 512, 1024) on retrieval quality
2. **Implement re-ranking**: Add cross-encoder for better relevance
3. **Build hybrid search**: Combine vector search with BM25